In [1]:
import sys
import subprocess
from bs4 import BeautifulSoup

In [2]:
name_map = {"Adidas": "Adidas+AG",
        "Nike": "Nike+Inc",
        "Puma": "Puma+SE",
        "Lululemon": "Lululemon+Athletica+Inc",
        "H & M": "H+&+M+Hennes+&+Mauritz+AB",
        "JD Sports": "JD+Sports+Fashion+PLC",
        "Gap": "Gap+Inc",
        "Burberry": "Burberry+Group+PLC",
        "Columbia": "Columbia+Sportswear+Co",
        "Boohoo": "Boohoo+Group+PLC",
        "Under Armour": "Under+Armour+Inc"
 }

companies = name_map.keys()

In [3]:
site = "https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data"

In [4]:
def get_html(url, selector=None, wait=False):

    cmd = [sys.executable, "webscrape.py", "--url", url]

    if selector:
        cmd += ["--selector", selector]

    if wait:
        cmd += ["--wait"]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("webscrape.py failed")

    with open("page.html", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    return soup



In [ ]:
ESG_data = {}
def build_ESG_data(ESG_data, company_name):
    full_site_link = site + "?esg=" + name_map[company_name]
    print(full_site_link)
    soup = get_html(
    full_site_link,
    selector="div.EsgRatings.EsgRatings--rating",
    wait=True
    )
    ESG_container = soup.select_one("div.EsgRatings.EsgRatings--rating")
    industry_div = soup.select_one("div.EsgRatings.EsgRatings--rank p")
    industry = industry_div.text.replace("Out of ", "").replace(" Companies.", "").strip()
    categories = ESG_container.find_all("div", class_="EsgRatings EsgRatings--category")
    ESG_data[company_name] = {
        "industry": industry
    }
    for category in categories:
        cat_name = category.find("h5").text
        ESG_data[company_name][cat_name]={}
        sub_cats= category.find_all("div", class_="EsgRatings EsgRatings--item")
        for sub_cat in sub_cats:
            sub_cat_name = sub_cat.find("div").text
            score = sub_cat.find("b").text
            ESG_data[company_name][cat_name][sub_cat_name]=score
    return ESG_data

for company in companies:
    try:
        ESG_data = build_ESG_data(ESG_data, company)
    except Exception as e:
        print(f"Failed for {company}: {e}")
        ESG_data[company] = {"error": str(e)}

print(ESG_data)

https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data?esg=Adidas+AG
https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data?esg=Nike+Inc
https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data?esg=Puma+SE
https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data?esg=Lululemon+Athletica+Inc
https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data?esg=H+&+M+Hennes+&+Mauritz+AB
[webscrape] Timeout on attempt 1/3 for https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data?esg=H+&+M+Hennes+&+Mauritz+AB
[webscrape] Timeout on attempt 2/3 for https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data?esg=H+&+M+Hennes+&+Mauritz+AB
[webscrape] Timeout on attempt 3/3 for https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-dat

RuntimeError: webscrape.py failed

In [ ]:
import pandas as pd

rows = []

for company, categories in ESG_data.items():
    industry = categories.get("industry")

    for category, metrics in categories.items():
        if category == "industry":
            continue

        for metric, score in metrics.items():
            rows.append({
                "company": company,
                "industry": industry,
                "category": category,
                "metric": metric,
                "score": score
            })

df = pd.DataFrame(rows)

In [ ]:
print(df)

   company       category                         metric score
0   Adidas  Environmental                  Environmental   3.2
1   Adidas  Environmental             Climate Transition     5
2   Adidas  Environmental          Energy & Resource Use     3
3   Adidas  Environmental                   Biodiversity     4
4   Adidas  Environmental                      Water Use     1
5   Adidas  Environmental              Waste & Pollution     2
6   Adidas         Social                         Social   3.8
7   Adidas         Social                Labour Reations     4
8   Adidas         Social                Health & Safety     3
9   Adidas         Social       Human Rights & Community     5
10  Adidas     Governance                     Governance   3.5
11  Adidas     Governance             Board & Management     3
12  Adidas     Governance             Shareholder Rights     4
13  Adidas     Governance      Conduct & Anti-Corruption     4
14  Adidas     Governance  Tax Transperancy & Accountin

In [ ]:
df.to_csv("esg_scores.csv", index=False)